---
# COSC2753 | Machine Learning

## Task 4 — Member 4: Advanced Fashion Visual Search
---

# 1. Introduction

This notebook develops and evaluates an image-retrieval system. Primary relevance is matching `articleType`; colour, gender, usage, and subcategory agreement are secondary diagnostics. The evaluation gallery uses frozen training rows and queries use internal-test rows, preventing same-group leakage.

# 2. Library Imports and Setup

In [ ]:
from copy import deepcopy
from pathlib import Path
import json, sys, time

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'scripts'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from skimage.feature import hog
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, Sampler
from torchvision import transforms

from preprocessing import IMAGE_SIZE, NORMALISATION_PATH, SEED, seed_everything, task_frame
seed_everything(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with NORMALISATION_PATH.open(encoding='utf-8') as handle:
    normalisation = json.load(handle)
DEVICE

# 3. Gallery and Query Protocol

In [ ]:
train_df = task_frame('articleType', 'train')
validation_df = task_frame('articleType', 'validation')
test_df = task_frame('articleType', 'test')
labels = sorted(train_df['articleType'].unique())
label_to_index = {label: index for index, label in enumerate(labels)}
len(train_df), len(validation_df), len(test_df), len(labels), set(train_df.group_key).intersection(test_df.group_key)

The final value above must be an empty set. During method selection, training rows are the gallery and test rows are queries. Only after evaluation is frozen do we build a deployment gallery from all valid supplied training images.

# 4. Image Dataset

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE[1], IMAGE_SIZE[0])), transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(8, translate=(0.05, 0.05)), transforms.ColorJitter(0.12, 0.12),
    transforms.ToTensor(), transforms.Normalize(normalisation['mean'], normalisation['std']),
])
evaluation_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE[1], IMAGE_SIZE[0])), transforms.ToTensor(),
    transforms.Normalize(normalisation['mean'], normalisation['std']),
])

class SearchDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame, self.transform = frame.reset_index(drop=True), transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row.image_path) as image:
            tensor = self.transform(image.convert('RGB'))
        return tensor, label_to_index[row.articleType]

class ClassAwareBatchSampler(Sampler):
    def __init__(self, target_values, batch_size=96, samples_per_class=4, seed=SEED):
        self.batch_size, self.samples_per_class, self.seed = batch_size, samples_per_class, seed
        self.classes_per_batch = batch_size // samples_per_class
        self.indices = {label: np.flatnonzero(np.asarray(target_values) == label) for label in np.unique(target_values)}
        self.classes = np.asarray(list(self.indices))
        self.sample_count = len(target_values)
    def __len__(self): return max(1, self.sample_count // self.batch_size)
    def __iter__(self):
        rng = np.random.default_rng(self.seed)
        for _ in range(len(self)):
            chosen = rng.choice(self.classes, self.classes_per_batch, replace=len(self.classes) < self.classes_per_batch)
            batch = []
            for label in chosen:
                candidates = self.indices[label]
                batch.extend(rng.choice(candidates, self.samples_per_class, replace=len(candidates) < self.samples_per_class).tolist())
            rng.shuffle(batch)
            yield batch

train_dataset = SearchDataset(train_df, train_transform)
train_sampler = ClassAwareBatchSampler(train_df.articleType.to_numpy())
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=0)
validation_loader = DataLoader(SearchDataset(validation_df, evaluation_transform), 128, num_workers=0)

# 5. Non-Learned Pixel Baseline

A small RGB thumbnail provides a transparent appearance baseline. It often retrieves similar colours/backgrounds but may miss semantic item type.

In [ ]:
def pixel_embedding(path, size=(12, 16)):
    with Image.open(path) as image:
        array = np.asarray(image.convert('RGB').resize(size), dtype=np.float32).reshape(-1) / 255.0
    return array / max(np.linalg.norm(array), 1e-12)

baseline_gallery = np.vstack([pixel_embedding(path) for path in train_df.image_path])
baseline_queries = np.vstack([pixel_embedding(path) for path in test_df.image_path])

In [ ]:
def retrieval_metrics(query_embeddings, gallery_embeddings, query_labels, gallery_labels, k_values=(1, 5, 10), batch_size=128):
    totals = {f'precision@{k}': 0.0 for k in k_values}
    totals.update({f'recall_hit@{k}': 0.0 for k in k_values})
    reciprocal_ranks = []
    for start in range(0, len(query_embeddings), batch_size):
        stop = min(start + batch_size, len(query_embeddings))
        scores = query_embeddings[start:stop] @ gallery_embeddings.T
        order = np.argsort(-scores, axis=1)
        batch_labels = query_labels[start:stop]
        for k in k_values:
            relevant = gallery_labels[order[:, :k]] == batch_labels[:, None]
            totals[f'precision@{k}'] += relevant.sum() / k
            totals[f'recall_hit@{k}'] += relevant.any(axis=1).sum()
        for query_offset, ranking in enumerate(order):
            matches = np.flatnonzero(gallery_labels[ranking] == batch_labels[query_offset])
            reciprocal_ranks.append(0.0 if len(matches) == 0 else 1.0 / (matches[0] + 1))
    metrics = {name: value / len(query_embeddings) for name, value in totals.items()}
    metrics['mean_reciprocal_rank'] = float(np.mean(reciprocal_ranks))
    return metrics

baseline_metrics = retrieval_metrics(
    baseline_queries, baseline_gallery, test_df.articleType.to_numpy(), train_df.articleType.to_numpy()
)
baseline_metrics

# 6. HOG + HSV Retrieval Baseline

In [ ]:
def handcrafted_embedding(path):
    with Image.open(path) as source:
        image = source.convert('RGB').resize((48, 64))
    array = np.asarray(image, dtype=np.float32) / 255.0
    shape = hog(array.mean(axis=2), orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2))
    hsv = np.asarray(image.convert('HSV'), dtype=np.float32) / 255.0
    colour = np.concatenate([np.histogram(hsv[..., channel], bins=8, range=(0, 1), density=True)[0] for channel in range(3)])
    feature = np.concatenate([shape, colour]).astype(np.float32)
    return feature / max(np.linalg.norm(feature), 1e-12)

handcrafted_gallery = np.vstack([handcrafted_embedding(path) for path in train_df.image_path])
handcrafted_queries = np.vstack([handcrafted_embedding(path) for path in test_df.image_path])
handcrafted_metrics = retrieval_metrics(
    handcrafted_queries, handcrafted_gallery, test_df.articleType.to_numpy(), train_df.articleType.to_numpy()
)
pd.DataFrame([baseline_metrics, handcrafted_metrics], index=['pixel', 'hog_hsv'])

# 7. Supervised-Contrastive Residual Encoder From Scratch

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, input_channels, output_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(input_channels, output_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(output_channels)
        self.conv2 = nn.Conv2d(output_channels, output_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(output_channels)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Identity() if input_channels == output_channels and stride == 1 else nn.Sequential(
            nn.Conv2d(input_channels, output_channels, 1, stride, bias=False), nn.BatchNorm2d(output_channels),
        )
    def forward(self, inputs):
        residual = self.shortcut(inputs)
        outputs = self.relu(self.bn1(self.conv1(inputs)))
        outputs = self.bn2(self.conv2(outputs))
        return self.relu(outputs + residual)

class EmbeddingCNN(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            ResidualBlock(3, 32), nn.MaxPool2d(2), ResidualBlock(32, 64), nn.MaxPool2d(2),
            ResidualBlock(64, 128), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, embedding_dim),
        )
    def forward(self, inputs):
        return nn.functional.normalize(self.features(inputs), dim=1)

model = EmbeddingCNN(128).to(DEVICE)
sum(parameter.numel() for parameter in model.parameters())

# 8. Contrastive Loss and Training

In [ ]:
def supervised_contrastive_loss(embeddings, targets, temperature=0.10):
    similarities = embeddings @ embeddings.T / temperature
    identity = torch.eye(len(targets), dtype=torch.bool, device=targets.device)
    positives = targets[:, None].eq(targets[None, :]) & ~identity
    valid = positives.sum(1) > 0
    if not valid.any():
        return embeddings.sum() * 0.0
    similarities = similarities - similarities.max(dim=1, keepdim=True).values.detach()
    exp_logits = similarities.exp().masked_fill(identity, 0)
    log_probabilities = similarities - torch.log(exp_logits.sum(1, keepdim=True).clamp_min(1e-12))
    per_anchor = -(log_probabilities * positives).sum(1) / positives.sum(1).clamp_min(1)
    return per_anchor[valid].mean()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def contrastive_epoch(loader, training=False):
    model.train(training); total_loss, samples = 0.0, 0
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            loss = supervised_contrastive_loss(model(images), targets)
            if training: loss.backward(); optimizer.step()
        total_loss += loss.item() * len(targets); samples += len(targets)
    return total_loss / samples

In [ ]:
history, best_state, best_loss, patience = [], None, float('inf'), 0
for epoch in range(1, 41):
    started = time.perf_counter()
    train_loss = contrastive_epoch(train_loader, True)
    validation_loss = contrastive_epoch(validation_loader)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'validation_loss': validation_loss})
    print(epoch, train_loss, validation_loss, f'{time.perf_counter() - started:.1f}s')
    if validation_loss < best_loss:
        best_loss, best_state, patience = validation_loss, deepcopy(model.state_dict()), 0
    else:
        patience += 1
        if patience >= 7: break
model.load_state_dict(best_state)
history = pd.DataFrame(history)
history.plot(x='epoch', y=['train_loss', 'validation_loss'], figsize=(8, 4)); plt.show()

## 8.1 Training observations

Discuss the class-aware sampler, convergence, overfitting, and the epoch selected using validation loss. Compare against random batches as a controlled ablation if time permits.

# 9. Learned Retrieval Evaluation

In [ ]:
@torch.inference_mode()
def embed_frame(frame):
    loader = DataLoader(SearchDataset(frame, evaluation_transform), 128, num_workers=0)
    model.eval(); batches = []
    for images, _ in loader:
        batches.append(model(images.to(DEVICE)).cpu().numpy())
    return np.vstack(batches)

gallery_embeddings = embed_frame(train_df)
query_embeddings = embed_frame(test_df)
encoder_metrics = retrieval_metrics(
    query_embeddings, gallery_embeddings, test_df.articleType.to_numpy(), train_df.articleType.to_numpy()
)
pd.DataFrame(
    [baseline_metrics, handcrafted_metrics, encoder_metrics],
    index=['pixel_baseline', 'hog_hsv_baseline', 'contrastive_encoder'],
)

# 10. Attribute Diagnostics

In [ ]:
top_five = np.vstack([np.argsort(-(query @ gallery_embeddings.T))[:5] for query in query_embeddings])
attribute_agreement = {}
for attribute in ('articleType', 'baseColour', 'gender', 'usage', 'subCategory'):
    query_values = test_df[attribute].to_numpy()[:, None]
    gallery_values = train_df[attribute].to_numpy()[top_five]
    attribute_agreement[attribute] = float((query_values == gallery_values).mean())
pd.Series(attribute_agreement, name='top_5_agreement')

# 11. Qualitative Retrieval Grid

In [ ]:
query_indices = np.linspace(0, len(test_df) - 1, 6, dtype=int)
fig, axes = plt.subplots(len(query_indices), 6, figsize=(12, 2.5 * len(query_indices)))
for row_number, query_index in enumerate(query_indices):
    scores = query_embeddings[query_index] @ gallery_embeddings.T
    retrieved_indices = np.argsort(-scores)[:5]
    items = [test_df.iloc[query_index], *[train_df.iloc[index] for index in retrieved_indices]]
    for column, item in enumerate(items):
        with Image.open(item.image_path) as image:
            axes[row_number, column].imshow(image.convert('RGB'))
        axes[row_number, column].set_title(('Query: ' if column == 0 else '') + item.articleType, fontsize=8)
        axes[row_number, column].axis('off')
plt.tight_layout()

## 10.1 Failure analysis

Use a fixed panel containing frequent, rare, visually confusable, and colour-dominated queries. Discuss semantic successes, background/colour shortcuts, failures, retrieval latency, gallery memory, and limits of defining similarity only through article type.

# 12. Save Encoder and Deployment Gallery

In [ ]:
checkpoint_path = ROOT / 'models' / 'visual_search_model.pt'
model = model.cpu()
torch.save({
    'state_dict': model.state_dict(), 'embedding_dim': 128,
    'mean': normalisation['mean'], 'std': normalisation['std'], 'image_size': list(IMAGE_SIZE),
    'selection_metric': 'validation_contrastive_loss', 'best_validation_loss': best_loss,
    'retrieval_metrics': encoder_metrics, 'seed': SEED,
}, checkpoint_path)
history.to_csv(ROOT / 'models' / 'visual_search_history.csv', index=False)

deployment_gallery = pd.concat([train_df, validation_df, test_df], ignore_index=True).drop_duplicates('id')
model = model.to(DEVICE)
deployment_embeddings = embed_frame(deployment_gallery)
np.save(ROOT / 'models' / 'visual_search_embeddings.npy', deployment_embeddings.astype('float32'))
deployment_gallery[['id', 'image_path', 'articleType', 'baseColour', 'gender', 'usage', 'subCategory']].to_csv(
    ROOT / 'models' / 'visual_search_metadata.csv', index=False
)
checkpoint_path, deployment_embeddings.shape

# 13. Final Search Check

Run `python scripts/task4_visual_search.py path/to/query.jpg --top-k 5` from the repository root after saving the encoder and gallery.

# 14. Ultimate Judgement and Conclusion

Replace this prompt with the measured pixel-versus-HOG+HSV-versus-encoder comparison, retrieval/attribute results, qualitative findings, efficiency, limitations, and a precise statement of what this system means by visually similar.